In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from lanxi_config import LanXiRecorderClient
import time

In [2]:
client = LanXiRecorderClient(ip_address="169.254.13.140")

In [3]:
# --- ГЛОБАЛЬНЫЕ НАСТРОЙКИ ---
w_rec_name  = widgets.Text(value='validation_test_run', description='Record Name:')
w_bandwidth = widgets.Dropdown(options=['51.2 kHz', '102.4 kHz', '25.6 kHz', '12.8 kHz'], value='51.2 kHz', description='Bandwidth:')
w_max_size  = widgets.Dropdown(options=[('2 GB', 2147483648), ('4 GB', 4294967296)], value=2147483648, description='Size Limit:')

# --- НАСТРОЙКИ КАНАЛОВ (1 - 4) ---
channels_widgets = {}
for i in range(1, 5):
    channels_widgets[i] = {
        "enabled": widgets.Checkbox(value=(i == 1), description=f"Enable Ch {i}"),
        "name":    widgets.Text(value=f"Channel {i}", description="Name:"),
        "range":   widgets.Dropdown(options=['10 Vpeak', '1 Vpeak', '0.316 Vpeak', '31.6 Vpeak'], value='10 Vpeak', description='Range:'),
        "filter":  widgets.Dropdown(options=['7.0 Hz', '0.7 Hz', 'DC', '22.4 Hz'], value='7.0 Hz', description='HP Filter:'),
        "ccld":    widgets.Checkbox(value=False, description='CCLD (IEPE) Power'),
        # Параметр моста добавлен для проверки расширяемого правила валидации
        "bridge_exc": widgets.Checkbox(value=(i == 1), description='Bridge Excitation (On)') if i == 1 else None
    }

# --- НАСТРОЙКИ ДАТЧИКА (ДЛЯ АКТИВНОГО КАНАЛА 1) ---
w_t_number  = widgets.Text(value='4961', description='Transd. No:')
w_t_serial  = widgets.IntText(value=3043249, description='Serial No:')
w_t_sens    = widgets.FloatText(value=0.05663, description='Sensitivity:', step=0.00001)
w_t_unit    = widgets.Dropdown(options=['Pa', 'V', 'm/s²', 'N'], value='Pa', description='Unit:')

btn_apply   = widgets.Button(description='Apply Parameters', button_style='success', icon='check', layout=widgets.Layout(width='200px', margin='10px 0px 0px 0px'))
out_panel   = widgets.Output()

# --- ВЕРСТКА ГРАФИЧЕСКОГО ИНТЕРФЕЙСА ---
box_global = widgets.VBox([widgets.HTML("<b>Global Settings</b>"), w_rec_name, w_bandwidth, w_max_size])
box_trans  = widgets.VBox([widgets.HTML("<b>Transducer Ch 1</b>"), w_t_number, w_t_serial, w_t_sens, w_t_unit])

# Собираем вкладки (Tabs) для независимого переключения между 4-мя каналами
ch_tabs_list = []
for i in range(1, 5):
    ch_w = channels_widgets[i]
    fields = [ch_w["enabled"], ch_w["name"], ch_w["range"], ch_w["filter"], ch_w["ccld"]]
    if ch_w["bridge_exc"]:
        fields.append(ch_w["bridge_exc"])
    ch_tabs_list.append(widgets.VBox(fields))

ch_tabs = widgets.Tab()
ch_tabs.children = ch_tabs_list
for i in range(4):
    ch_tabs.set_title(i, f"Channel {i+1}")

form_ui = widgets.VBox([
    widgets.HBox([box_global, ch_tabs, box_trans], layout=widgets.Layout(justify_content='space-between', padding='10px')),
    btn_apply
])

def on_apply_clicked(b):
    with out_panel:
        clear_output()
        
        # Динамически собираем структуру JSON из интерфейса для всех 4 каналов
        channels_list = []
        for i in range(1, 5):
            ch_w = channels_widgets[i]
            ch_dict = {
                "channel": i,
                "name": ch_w["name"].value,
                "enabled": ch_w["enabled"].value,
                "range": ch_w["range"].value,
                "filter": ch_w["filter"].value,
                "floating": False,
                "ccld": ch_w["ccld"].value,
                "destinations": ["sd"],
                "bandwidth": w_bandwidth.value
            }
            
            # Добавляем специфичные для моста параметры
            if ch_w["bridge_exc"]:
                ch_dict["bridgeExcOn"] = ch_w["bridge_exc"].value
                
            # Привязываем объект transducer к первому каналу
            if i == 1:
                ch_dict["transducer"] = {
                    "requiresCcld": ch_w["ccld"].value,
                    "requires200V": False,
                    "sensitivity": w_t_sens.value,
                    "serialNumber": w_t_serial.value,
                    "type": {"model": "", "number": w_t_number.value, "prefix": "", "variant": ""},
                    "unit": w_t_unit.value
                }
            channels_list.append(ch_dict)

        config_payload = {
            "name": w_rec_name.value,
            "fileFormat": "wav",
            "maxSize": w_max_size.value,
            "recordingMode": "Single",
            "channels": channels_list
        }
        
        # Запуск жесткого сброса и попытка применить конфигурацию через FSM
        client.force_reset_to_idle()
        time.sleep(1)
        
        if client.apply_configuration(config_payload):
            client.display_current_state()
        else:
            client.display_current_state()

btn_apply.on_click(on_apply_clicked)
display(form_ui, out_panel)

Output()

In [4]:
# Write config to SD
if client.start_recording():
    time.sleep(2)
    client.stop_recording()

client.stop_and_close()

2026-10-16 16:58:47: FSM Blocked 'start_recording': state is 'RecorderConfiguring', but must be one of ['RecorderStreaming']
2026-10-16 16:58:47: Finishing current recording session...


False

In [12]:
client.stop_recording()
if not client.stop_and_close():
    client.force_reset_to_idle()

2026-10-16 17:01:54: FSM Blocked 'stop_recording': state is 'RecorderConfiguring', but must be one of ['RecorderRecording']
2026-10-16 17:01:54: Finishing current recording session...
2026-10-16 17:01:54: Initiating forced hard reset to Idle state...
2026-10-16 17:01:54: Device successfully forced into Idle state.
2026-10-16 17:01:54: Current FSM confirmed state: Idle
